# SAA Challenger Model (UKRM)

Independent challenger for the Strategic Asset Allocation (SAA) model. It reproduces the model's three outputs for each strategy:

| Block | What the SAA Model does | Main output |
|---|---|---|
| **A. Optimisation** | RISKOptimizer chooses the composite weights that maximise the mean portfolio value in year 20 (cell W97), subject to the strategy's constraints. The composite weights are then unbundled into the 19 indices. | Composite weights per strategy, and index-level weights |
| **B. Historic simulation** | The optimised portfolios are back-tested on the last 20 years of monthly data. | Historic Characteristics per strategy |
| **C. Monte Carlo simulation** | 10,000 simulated 20-year paths show how each portfolio might behave in future. | Projected Characteristics per strategy |

Block A is solved two ways:
- **Replica** – the same recipe as RISKOptimizer: simulate, score each candidate portfolio (mean year-20 value minus penalties), and search with a genetic-style algorithm.
- **Analytical** – a direct calculation with no simulation: the highest expected return that meets every constraint exactly. Because mean year-20 value = starting value × (1 + expected return)²⁰, both are aiming at the same answer.

Notebook sections: **1. Setup · 2. Input Data Reading · 3. Block A – Optimisation · 4. Block B – Historic Characteristics · 5. Block C – Projected Characteristics · 6. Output Creation**

## 1. Setup
Edit the cells in this section only.

In [ ]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import saa_helpers as h          # helper functions (saa_helpers.py, same folder)

print("numpy", np.__version__, "| scipy", scipy.__version__, "| pandas", pd.__version__)

def pct(v, d=2):
    """Show numbers as percentages, leave text (e.g. dates) unchanged."""
    return ("{:." + str(d) + "%}").format(v) if isinstance(v, (float, np.floating)) and not np.isnan(v) else v

def show(df, d=2):
    """Return a copy of a table with numbers shown as percentages."""
    f = df.applymap if hasattr(df, "applymap") else df.map     # works on old and new pandas
    return f(lambda v: pct(v, d))

In [ ]:
# --- Files ---
INPUT_FILE  = "SAA_Challenger_Inputs_v2.xlsx"
OUTPUT_FILE = "SAA_Challenger_Results.xlsx"

# --- Strategies, in the order they are optimised (Core first, then Income and Responsible) ---
# None = all columns of the Setup tab, in the order they appear there.
STRATEGIES = None

# --- Volatility targets ---
# True: targets evenly spaced from 30% to 90% of Equity volatility, using the risk levels below.
# False: use the 'Annualised Volatility' row typed in the Setup tab.
AUTO_VOL_TARGETS = True
N_RISK_LEVELS = 4
RISK_LEVELS = {                      # PLACEHOLDER mapping - confirm with the model owner
    "Defensive": 1, "Cautious": 2, "Balanced": 3, "Growth": 4,
    "Income": 2, "Income Plus": 3,
    "Defensive Responsible": 1, "Cautious Responsible": 2,
    "Balanced Responsible": 3, "Growth Responsible": 4,
}

# --- Optimisation (Block A) ---
GENERATIONS = 300          # search rounds per pass; each tests (PopulationPerAsset x 7) portfolios
VOL_MEASURE = "pooled"     # how the Replica measures portfolio volatility: "pooled" or "mean_path"

# --- Historic simulation (Block B) ---
HISTORIC_YEARS = 20
REBALANCE = "monthly"      # "monthly" or "annual"

# --- Monte Carlo (Block C) ---
FINAL_ITERATIONS = 10000

# --- Developer's composite weights, for comparison (optional) ---
# Order: Equity, DMGovt, BondOther, Property, Gold, HFs, Cash  (decimals, summing to 1)
DEVELOPER_WEIGHTS = {
    # "Balanced": [0.60, 0.05, 0.20, 0.05, 0.05, 0.03, 0.02],
}

## 2. Input Data Reading

In [ ]:
inputs = h.read_inputs(INPUT_FILE)
st = inputs["settings"]
strategies = STRATEGIES or list(inputs["setup"].columns)

h.check_inputs(inputs)
print()
print("Starting portfolio value:", st["InitialValue"], "| Horizon:", st["HorizonYears"], "years",
      "| Distribution:", st["Distribution"])

In [ ]:
summary_inputs = pd.DataFrame({
    "Expected return": inputs["returns"],
    "Volatility": inputs["vols"],
    "Yield": inputs["yields"],
}).loc[h.ALL_VARS]
summary_inputs.pipe(show)

In [ ]:
print("Correlation matrix (treated as rank / Spearman correlation, as in @RISK):")
inputs["corr"].round(2)

In [ ]:
print("Constraints per strategy (from the Setup tab):")
inputs["setup"].pipe(show, 1)

## 3. Block A – Optimisation → Composite Output

**Main output:** the weight of each asset-class composite for each strategy, and the same weights unbundled to the 19 indices.

### 3.1 Volatility targets

In [ ]:
if AUTO_VOL_TARGETS:
    vol_targets = h.volatility_targets(inputs, RISK_LEVELS, N_RISK_LEVELS)
    print("Targets set at 30%-90% of Equity volatility ({:.2%}):".format(inputs["vols"]["Equity"]))
else:
    vol_targets = inputs["setup"].loc["Annualised Volatility"]
    print("Targets taken from the Setup tab:")
vol_targets[strategies].map(pct).to_frame("Volatility target")

### 3.2 Replica optimisation (RISKOptimizer recipe)
Two passes per strategy, as in the SAA Instructions: 500 simulated iterations, then 1,000 new iterations starting from the pass-1 answer. Takes roughly 5–10 seconds per strategy.

In [ ]:
replica = {}
for s in strategies:
    print("=" * 70)
    print("BLOCK A | REPLICA | {}  (volatility target {:.2%})".format(s, vol_targets[s]))
    cons = inputs["setup"][s]

    print("   Pass 1: {} iterations ...".format(int(st["Pass1Iterations"])))
    raw1, w1 = h.optimise_replica(inputs, cons, vol_targets[s], int(st["Pass1Iterations"]),
                                  seed=1, generations=GENERATIONS, vol_measure=VOL_MEASURE)
    print("           raw weights (0-1000):", raw1.astype(int))

    print("   Pass 2: {} iterations, starting from pass 1 ...".format(int(st["Pass2Iterations"])))
    raw2, w2 = h.optimise_replica(inputs, cons, vol_targets[s], int(st["Pass2Iterations"]),
                                  seed=2, start=raw1, generations=GENERATIONS, vol_measure=VOL_MEASURE)
    print("           raw weights (0-1000):", raw2.astype(int))
    replica[s] = w2
print("Replica optimisation complete.")

### 3.3 Analytical optimisation (benchmark)

In [ ]:
analytical = {}
for s in strategies:
    print("BLOCK A | ANALYTICAL | {}".format(s))
    w = h.optimise_analytical(inputs, inputs["setup"][s], vol_targets[s])
    if w is not None:
        analytical[s] = w
print("Analytical optimisation complete.")

### 3.4 Composite output
One table per method: rows = strategies, columns = composites (the layout of the SAA Optimisation document).

In [ ]:
composite = {
    "Replica": pd.DataFrame(replica, index=h.ASSETS).T,
    "Analytical": pd.DataFrame(analytical, index=h.ASSETS).T,
}
if DEVELOPER_WEIGHTS:
    composite["Developer"] = pd.DataFrame(DEVELOPER_WEIGHTS, index=h.ASSETS).T

for method, table in composite.items():
    print("Composite weights -", method)
    display(table.pipe(show, 1))

In [ ]:
print("Difference: Replica minus Analytical (percentage points)")
(composite["Replica"] - composite["Analytical"]).pipe(show, 1)

In [ ]:
if "Developer" in composite:
    print("Difference: Developer minus Analytical (percentage points)")
    display((composite["Developer"] - composite["Analytical"]).dropna().pipe(show, 1))

In [ ]:
fig, axes = plt.subplots(1, len(composite), figsize=(6 * len(composite), 5), sharey=True)
axes = np.atleast_1d(axes)
for ax, (method, table) in zip(axes, composite.items()):
    table.plot(kind="bar", stacked=True, ax=ax, legend=False, colormap="tab10")
    ax.set_title(method)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: "{:.0%}".format(y)))
axes[-1].legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 3.5 Unbundling to the 19 indices

In [ ]:
index_level = {method: h.unbundle(table, inputs["asset_map"]) for method, table in composite.items()}
for method, table in index_level.items():
    print("Index-level weights -", method)
    display(table.T.pipe(show, 2))

## 4. Block B – Historic Simulation → Historic Characteristics

**Main output:** for each strategy, how the optimised portfolio would have behaved over the last 20 years of monthly data.
Requires the `HistoricReturns` tab (and `HistoricYields` for the average yield).

In [ ]:
historic = {}
if inputs["hist_returns"] is None:
    print("No HistoricReturns tab - Block B skipped.")
else:
    for method, table in composite.items():
        print("BLOCK B | HISTORIC SIMULATION |", method)
        rows = {s: h.historic_characteristics(table.loc[s].values, inputs, HISTORIC_YEARS, REBALANCE)
                for s in table.index}
        historic[method] = h.to_table(rows)
    for method, table in historic.items():
        print("Historic Characteristics (over {} years) - {}".format(HISTORIC_YEARS, method))
        display(table.pipe(show))

## 5. Block C – Monte Carlo Simulation → Projected Characteristics

**Main output:** for each strategy, the range of outcomes over 20 simulated years.
All portfolios use the **same** 10,000 simulated paths, so differences come from the weights only.

In [ ]:
print("BLOCK C | MONTE CARLO | simulating {} paths of {} years ...".format(FINAL_ITERATIONS, int(st["HorizonYears"])))
final_sims = h.simulate_returns(inputs, FINAL_ITERATIONS, int(st["HorizonYears"]),
                                seed=99, distribution=st["Distribution"])

projected = {}
for method, table in composite.items():
    print("BLOCK C | PROJECTED CHARACTERISTICS |", method)
    rows = {s: h.projected_characteristics(table.loc[s].values, final_sims, inputs) for s in table.index}
    projected[method] = h.to_table(rows)
for method, table in projected.items():
    print("Projected Characteristics - {}".format(method))
    display(table.pipe(show))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for a in h.ASSETS:
    ax.scatter(inputs["vols"][a], inputs["returns"][a], color="grey", marker="s")
    ax.annotate(a, (inputs["vols"][a], inputs["returns"][a]), fontsize=8, xytext=(4, 2), textcoords="offset points")
markers = {"Replica": "o", "Analytical": "x", "Developer": "*"}
cov = h.covariance_matrix(inputs)
for method, table in composite.items():
    vols = [np.sqrt(w @ cov @ w) for w in table.values]
    rets = table.values @ inputs["returns"][h.ASSETS].values
    ax.scatter(vols, rets, marker=markers[method], s=70, label=method)
ax.legend()
ax.set_xlabel("Expected volatility")
ax.set_ylabel("Expected return")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: "{:.0%}".format(x)))
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: "{:.1%}".format(y)))
ax.set_title("Composites and optimised strategies")
plt.show()

## 6. Output Creation

In [ ]:
sheets = {"Inputs": summary_inputs, "VolTargets": vol_targets.to_frame("Volatility target")}
for method in composite:
    sheets["Composite_" + method] = composite[method]
    sheets["Index_" + method] = index_level[method].T
    if method in historic:
        sheets["Historic_" + method] = historic[method]
    sheets["Projected_" + method] = projected[method]
h.save_results(OUTPUT_FILE, sheets)